# 02 - Train the span-based SciBERT NER model

Enumerates all spans of up to 8 tokens per sentence and classifies each into one of the six
SciERC entity types or `O`. Nested entities are supported because spans are classified
independently. Training is resumable: checkpoints (model, optimiser, scheduler, RNG state)
are written to `OUTPUTS/ner/checkpoints/last.pt` and re-running the training cell continues
from the last checkpoint.

The configuration `configs/ner.yaml` targets a GPU (cross-sentence context, 15 epochs);
`configs/ner_cpu.yaml` is the reduced configuration used for the CPU run reported in the
README.

In [ ]:
# Environment setup: works on Kaggle, Google Colab and a local checkout.
import os, subprocess, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HaoudarNouhaila/scigraph-nlp.git"

if IN_KAGGLE or IN_COLAB:
    base = Path("/kaggle/working") if IN_KAGGLE else Path("/content")
    ROOT = base / "scigraph-nlp"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("project root:", ROOT)

In [ ]:
# Download the official SciERC release (once) and build sentence-level splits.
if not Path("data/processed/train.jsonl").exists():
    if not Path("data/raw/scierc/train.json").exists():
        subprocess.run([sys.executable, "scripts/download_scierc.py", "--out", "data/raw/scierc"], check=True)
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
print(sorted(p.name for p in Path("data/processed").glob("*.jsonl")))

In [ ]:
# Persistent output directory so that an interrupted session can resume training.
# Colab: Google Drive. Kaggle: attach the output of a previous run of this notebook as an
# input dataset and it is copied back before training starts.
import shutil
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUTS = Path("/content/drive/MyDrive/scigraph-nlp/outputs")
else:
    OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)
if IN_KAGGLE:
    for prev in Path("/kaggle/input").glob("*/outputs"):
        print("restoring checkpoints from", prev)
        shutil.copytree(prev, OUTPUTS, dirs_exist_ok=True)
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import yaml
CONFIG = "configs/ner.yaml" if torch.cuda.is_available() else "configs/ner_cpu.yaml"
print(yaml.safe_load(Path(CONFIG).read_text()))

In [ ]:
# Training (resumes automatically if a checkpoint exists in OUTPUTS/ner).
cmd = [sys.executable, "-m", "scigraph.training.train_ner", "--config", CONFIG,
       "--set", f"output_dir={OUTPUTS / 'ner'}"]
subprocess.run(cmd, check=True)

In [ ]:
import json
m = json.loads((OUTPUTS / "ner" / "metrics.json").read_text())
print("dev  F1:", m["dev"]["micro"]["f1"])
print("test P/R/F1:", m["test"]["micro"]["precision"], m["test"]["micro"]["recall"], m["test"]["micro"]["f1"])
import pandas as pd
pd.DataFrame(m["test"]["per_type"]).T